# MicroDuck 直播 Notebook：球平衡 / FastSAC

把任务、策略、ONNX、MuJoCo 和 RDK X5/BPU 验收串成一条可复用流程。

## 0. 运行说明

建议在 `02-可运行代码/microduck-playground-stilts` 的 Python 环境中启动 Jupyter。训练模型和板端 HBM 不随 Git 提交；通过 `MICRODUCK_ONNX`、`RDK_BPU_HBM` 和 `RDK_HOST` 注入。

In [ ]:
from pathlib import Path
import json
import os
import platform
import shlex
import socket
import subprocess


def locate_topic_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "02-可运行代码").is_dir() and (base / "01-任务资料").is_dir():
            return base
        group = base / "16-专题组队学习"
        if group.is_dir():
            matches = list(group.glob("*/02-可运行代码"))
            if matches:
                return matches[0].parent
    configured = os.getenv("MICRODUCK_TOPIC_ROOT")
    if configured and Path(configured).is_dir():
        return Path(configured).resolve()
    raise RuntimeError("找不到专题目录，请从 03-Notebook 启动 Jupyter，或设置 MICRODUCK_TOPIC_ROOT。")


TOPIC_ROOT = locate_topic_root()
PLAYGROUND_ROOT = TOPIC_ROOT / "02-可运行代码" / "microduck-playground-stilts"
OUTPUT_ROOT = TOPIC_ROOT / "03-Notebook" / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("专题目录:", TOPIC_ROOT)
print("可运行代码:", PLAYGROUND_ROOT)
print("输出目录:", OUTPUT_ROOT)
print("Jupyter 工作站:", socket.gethostname(), platform.system(), platform.machine())
print("RDK 地址:", os.getenv("RDK_HOST", "192.168.8.128"), "用户: sunrise")

## 1. 模型契约与本地检查

目标契约是 actor `obs[1, 54] -> action[1, 14]`。如果本任务没有随专题提交 checkpoint/ONNX，Notebook 会明确显示“模型未提供”，不会用别的任务模型冒充当前任务结果。

In [ ]:
TASK_ID = 'microduck-ball-balance'
MODEL_HINT = None
MODEL_PATH = Path(os.getenv("MICRODUCK_ONNX", "")) if os.getenv("MICRODUCK_ONNX") else None
if MODEL_PATH is None and MODEL_HINT:
    candidate = TOPIC_ROOT / MODEL_HINT
    if candidate.exists():
        MODEL_PATH = candidate
if MODEL_PATH is None and not MODEL_HINT and not TASK_ID:
    candidates = sorted(TOPIC_ROOT.glob("01-任务资料/**/*.onnx"))
    MODEL_PATH = candidates[0] if candidates else None
print("任务:", TASK_ID or "接口/网页演示")
print("ONNX:", MODEL_PATH if MODEL_PATH else "未提供")
if TASK_ID and MODEL_PATH is None:
    print("该任务尚未随专题提交任务专属 ONNX；请设置 MICRODUCK_ONNX 后再运行模型检查。")

In [ ]:
def inspect_onnx(path):
    try:
        import onnx
    except ImportError:
        print("缺少 onnx：请在 microduck-playground 环境中启动 Jupyter。")
        return None
    model = onnx.load(str(path))
    onnx.checker.check_model(model)
    def shape(value):
        return [d.dim_value if d.dim_value else (d.dim_param or "?") for d in value.type.tensor_type.shape.dim]
    result = {
        "inputs": [(item.name, shape(item)) for item in model.graph.input],
        "outputs": [(item.name, shape(item)) for item in model.graph.output],
        "nodes": len(model.graph.node),
        "metadata": {item.key: item.value for item in model.metadata_props},
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

contract = inspect_onnx(MODEL_PATH) if MODEL_PATH else None

## 2. ONNX 导出入口

训练得到的是策略 checkpoint；导出时要把 actor 和观测归一化一起固化到 ONNX。下面是本任务命令模板。`--video` 会在本机录制仿真回放，完整视频写入 `outputs/`，不提交到 Git。

```bash
python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11
```

In [ ]:
EXPORT_COMMAND = "python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11" if TASK_ID == "microduck-ball-balance" else ("uv run python scripts/export.py " + (TASK_ID or "<TASK_ID>") + " --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250")
print(EXPORT_COMMAND)
print("导出前后检查：观测维度、动作顺序、归一化和 action clip。")

## 3. FastSAC smoke 训练与 ONNX 导出

球平衡使用 MotrixLab 的 FastSAC，不复用 PPO/LSTM。默认在 Ubuntu 的 MotrixLab `.venv` 中用 64 个并行环境做 5 个 smoke iteration，并把本次 run 导出为 `outputs/motrix_ball_balance_latest.onnx`。若重新训练后要生成新的板端视频，需要同步重新执行 X5 HBM 编译单元，不能继续使用旧 HBM。

In [ ]:
import shutil
import sys

MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser()
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser()
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py"
FASTSAC_ENVS = int(os.getenv("MICRODUCK_FASTSAC_ENVS", "64"))
FASTSAC_ITERATIONS = int(os.getenv("MICRODUCK_FASTSAC_ITERATIONS", "5"))
RUN_FASTSAC_SMOKE = os.getenv("MICRODUCK_RUN_FASTSAC_SMOKE", "1") == "1"
FASTSAC_ONNX = OUTPUT_ROOT / "motrix_ball_balance_latest.onnx"
print("MotrixLab:", MOTRIX_ROOT)
print("FastSAC Python:", MOTRIX_PYTHON)
print("固定输出:", FASTSAC_ONNX)

if not RUN_FASTSAC_SMOKE:
    print("已跳过 FastSAC smoke 训练：设置 MICRODUCK_RUN_FASTSAC_SMOKE=1 后重新运行。")
elif not MOTRIX_ROOT.is_dir() or not MOTRIX_PYTHON.is_file():
    print("找不到 MotrixLab 或其 Python 环境；请先在 Ubuntu 完成 MotrixLab 环境安装。")
else:
    train_cmd = [
        str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "train", str(MOTRIX_ROOT / "scripts/train.py"),
        "task=microduck-ball-balance/motrix.fastsac",
        f"num_envs={FASTSAC_ENVS}",
        "play=false", "render=false", "algo.asynchronous=false",
        "algo.device=cuda", "algo.agent.compile=false", "algo.agent.amp=false",
        # Keep the default run genuinely smoke-sized. The replay buffer must
        # be populated before FastSAC can take its first update.
        "algo.agent.learning_starts=0", "algo.agent.buffer_size=8", "algo.agent.batch_size=64",
        "algo.agent.num_updates=1", f"algo.trainer.num_learning_iterations={FASTSAC_ITERATIONS}",
    ]
    print("开始 FastSAC smoke 训练:", " ".join(shlex.quote(x) for x in train_cmd))
    train_result = subprocess.run(train_cmd, cwd=MOTRIX_ROOT, text=True)
    if train_result.returncode != 0:
        raise RuntimeError(f"FastSAC smoke 训练失败，returncode={train_result.returncode}")
    run_root = MOTRIX_ROOT / "runs" / "microduck-ball-balance" / "motrix" / "torch" / "fastsac"
    run_dirs = sorted((p for p in run_root.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        raise FileNotFoundError(f"训练成功但没有找到 FastSAC run: {run_root}")
    run_dir = run_dirs[-1]
    motrix_export = Path("outputs") / "motrix_ball_balance_latest.onnx"
    export_cmd = [
        str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "export", str(MOTRIX_ROOT / "scripts/export_onnx.py"),
        f"run_dir={run_dir}", f"output={motrix_export.as_posix()}", "opset=11",
    ]
    print("导出 FastSAC ONNX:", " ".join(shlex.quote(x) for x in export_cmd))
    export_result = subprocess.run(export_cmd, cwd=MOTRIX_ROOT, text=True)
    exported_onnx = MOTRIX_ROOT / motrix_export
    if export_result.returncode != 0 or not exported_onnx.is_file():
        raise RuntimeError("FastSAC ONNX 导出失败。")
    shutil.copy2(exported_onnx, FASTSAC_ONNX)
    MODEL_PATH = FASTSAC_ONNX
    print("PASS-fastsac-model:", MODEL_PATH)
    print("实际训练 run:", run_dir)

## 3b. X5 HBM 编译

训练后的 ONNX 不能直接交给 RDK X5。下面固定输入输出 batch 为 1，并通过 Ubuntu 上的官方 X5 CPU Docker 工具链运行 `hb_mapper makertbin --model-type onnx`。生成的 `ball_balance.opset11.bin` 会覆盖编译目录，下一节视频使用的就是这份新 HBM。

In [ ]:
import shutil

BPU_MODEL_ROOT = Path(os.getenv("MICRODUCK_BPU_MODEL_ROOT", "/home/ubuntu/workspaces/microduck_bpu_models")).expanduser()
BPU_CONVERTED = BPU_MODEL_ROOT / "converted"
BPU_COMPILED = BPU_MODEL_ROOT / "compiled" / "ball_balance"
BPU_STATIC_ONNX = BPU_CONVERTED / "motrix_ball_balance.static.opset11.onnx"
BPU_CONFIG = BPU_COMPILED / ".fast_perf" / "ball_balance.opset11_config.yaml"
BPU_OUTPUT = BPU_COMPILED / "model_output" / "ball_balance.opset11.bin"
BPU_DOCKER_IMAGE = os.getenv("MICRODUCK_BPU_DOCKER_IMAGE", "openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8")

def workspace_container_path(path):
    workspace = Path("/home/ubuntu/workspaces")
    try:
        return Path("/work") / path.resolve().relative_to(workspace)
    except ValueError as exc:
        raise RuntimeError(f"HBM 文件必须位于 {workspace}: {path}") from exc

if not FASTSAC_ONNX.is_file():
    print("跳过 HBM 编译：先成功运行第 3 节 FastSAC 训练与导出。")
elif shutil.which("docker") is None:
    print("跳过 HBM 编译：当前 Ubuntu 找不到 docker。")
else:
    import onnx

    BPU_CONVERTED.mkdir(parents=True, exist_ok=True)
    BPU_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    BPU_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    for stale_file in BPU_OUTPUT.parent.iterdir():
        if stale_file.is_file():
            stale_file.unlink()
    static_model = onnx.load(str(FASTSAC_ONNX))
    for value in list(static_model.graph.input) + list(static_model.graph.output):
        value.type.tensor_type.shape.dim[0].dim_value = 1
        value.type.tensor_type.shape.dim[0].ClearField("dim_param")
    onnx.checker.check_model(static_model)
    onnx.save(static_model, str(BPU_STATIC_ONNX))
    BPU_CONFIG.write_text(
        """calibration_parameters:
  cal_data_type: ''
  calibration_type: skip
  optimization: run_fast
  per_channel: false
  preprocess_on: false
  run_on_bpu: ''
  run_on_cpu: ''
compiler_parameters:
  compile_mode: latency
  core_num: 1
  debug: true
  jobs: 0
  max_time_per_fc: 0
  optimize_level: O3
input_parameters:
  input_layout_rt: NCHW
  input_layout_train: NCHW
  input_name: obs
  input_shape: 1x54
  input_space_and_range: ''
  input_type_rt: featuremap
  input_type_train: featuremap
  norm_type: no_preprocess
model_parameters:
  layer_out_dump: false
  march: bayes-e
  onnx_model: /work/microduck_bpu_models/converted/motrix_ball_balance.static.opset11.onnx
  output_model_file_prefix: ball_balance.opset11
  remove_node_type: Quantize;Transpose;Dequantize;Cast;Reshape;Softmax;DequantizeFilter
  set_node_data_type: {}
  working_dir: /work/microduck_bpu_models/compiled/ball_balance/model_output
""",
        encoding="utf-8",
    )
    docker_cmd = [
        "docker", "run", "--rm", "-v", "/home/ubuntu/workspaces:/work",
        BPU_DOCKER_IMAGE, "sh", "-lc",
        "hb_mapper makertbin --config "
        + shlex.quote(str(workspace_container_path(BPU_CONFIG)))
        + " --model-type onnx",
    ]
    print("开始 X5 HBM 编译:", " ".join(shlex.quote(x) for x in docker_cmd))
    compile_result = subprocess.run(docker_cmd, text=True)
    if compile_result.returncode != 0 or not BPU_OUTPUT.is_file():
        raise RuntimeError("X5 HBM 编译失败。")
    os.environ["MICRODUCK_BPU_HBM"] = str(BPU_OUTPUT)
    print("PASS-X5-HBM:", BPU_OUTPUT)

## 4. 本地 ONNX 基准推理

这一步只验证 ONNX 图能在本机运行，以及输出形状和数值是否有限；它不是训练效果评测，也不是 BPU 验收。

In [ ]:
def onnx_local_smoke(path):
    try:
        import numpy as np
        import onnxruntime as ort
    except ImportError as exc:
        print("缺少本地推理依赖:", exc)
        return None
    session = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    feeds = {}
    for item in session.get_inputs():
        shape = [dim if isinstance(dim, int) and dim > 0 else 1 for dim in item.shape]
        feeds[item.name] = np.zeros(shape, dtype=np.float32)
    outputs = session.run(None, feeds)
    info = {"providers": session.get_providers(), "inputs": [(x.name, x.shape) for x in session.get_inputs()], "outputs": [(x.name, x.shape) for x in session.get_outputs()], "finite": all(np.isfinite(x).all() for x in outputs)}
    print(json.dumps(info, ensure_ascii=False, indent=2, default=str))
    return outputs

if MODEL_PATH:
    local_outputs = onnx_local_smoke(MODEL_PATH)
else:
    print("跳过：尚未提供该任务 ONNX。")

## 5. RDK X5 BPU-in-the-loop 视频

本单元只展示新生成的板端闭环 MP4，不读取参考 GIF 或关键帧。Ubuntu 运行 MuJoCo 物理和 EGL 渲染；每一个控制步把任务观测发送到 RDK，RDK 用当前任务对应的 HBM 在 BPU 上计算 14 维动作，再返回给 Ubuntu。视频因此是“板端 BPU 产生动作、Ubuntu 负责仿真画面”的真实闭环。

运行前会自动重启 RDK 上的任务策略服务。切换 Notebook 时必须重新运行本单元，让板端 HBM 与当前任务匹配。

In [ ]:
import hashlib
import sys

BPU_TASK = 'ball_balance'
BPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_BPU_VIDEO_STEPS", "250"))
BPU_VIDEO_WIDTH = int(os.getenv("MICRODUCK_BPU_VIDEO_WIDTH", "640"))
BPU_VIDEO_HEIGHT = int(os.getenv("MICRODUCK_BPU_VIDEO_HEIGHT", "480"))
BPU_PORT = int(os.getenv("MICRODUCK_BPU_PORT", "8765"))
RDK_CONTROL_PORT = int(os.getenv("MICRODUCK_BPU_CONTROL_PORT", "8766"))
RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
BPU_HBM_DEFAULTS = {
    "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ball_balance/model_output/ball_balance.opset11.bin",
    "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/basketball/model_output/basketball.opset11.bin",
    "walking": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
    "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
    "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/stilts25/model_output/stilts25.opset11.bin",
    "swing": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/swing/model_output/swing.opset11.bin",
    "ladder": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ladder/model_output/ladder.opset11.bin",
}
RDK_HBM_DEFAULTS = {
    "ball_balance": "/home/sunrise/microduck_policy/hbm/ball_balance.bin",
    "basketball": "/home/sunrise/microduck_policy/hbm/basketball.bin",
    "walking": "/home/sunrise/microduck_policy/hbm/walking.bin",
    "perturbation": "/home/sunrise/microduck_policy/hbm/walking.bin",
    "stilt": "/home/sunrise/microduck_policy/hbm/stilts25.bin",
    "swing": "/home/sunrise/microduck_policy/hbm/swing.bin",
    "ladder": "/home/sunrise/microduck_policy/hbm/ladder.bin",
}
BPU_HBM = Path(os.getenv("MICRODUCK_BPU_HBM", BPU_HBM_DEFAULTS.get(BPU_TASK, ""))) if BPU_TASK else None
RDK_HBM = RDK_HBM_DEFAULTS.get(BPU_TASK, "") if BPU_TASK else ""
RDK_SERVER_SCRIPT = os.getenv("RDK_BPU_SERVER_SCRIPT", "/home/sunrise/microduck_policy/rdk_bpu_policy_server.py")
MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if BPU_TASK == "ball_balance" else None
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if BPU_TASK == "ball_balance" else None
BPU_VIDEO_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_bpu_inloop_video.py" if BPU_TASK == "ball_balance" else "bpu_inloop_video.py")
BPU_VIDEO_OUTPUT = OUTPUT_ROOT / "bpu_videos" / f"{BPU_TASK}_bpu_latest.mp4" if BPU_TASK else None

def restart_rdk_policy_server():
    if not BPU_TASK or not RDK_HBM:
        return False
    try:
        if BPU_HBM is not None and BPU_HBM.is_file():
            payload = BPU_HBM.read_bytes()
            request = {
                "op": "upload_and_switch",
                "task": BPU_TASK,
                "size": len(payload),
                "sha256": hashlib.sha256(payload).hexdigest(),
            }
        else:
            payload = b""
            request = {"op": "switch", "task": BPU_TASK}
        with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
            control.sendall((json.dumps(request) + "\n").encode("utf-8"))
            if payload:
                for offset in range(0, len(payload), 1024 * 1024):
                    control.sendall(payload[offset:offset + 1024 * 1024])
            response = json.loads(control.makefile("rb").readline().decode("utf-8"))
        if response.get("ok"):
            action = "已上传并切换" if payload else "已切换"
            print("RDK 任务策略服务" + action + ":", BPU_TASK, "->", response.get("model"), "pid=", response.get("pid"))
            return True
        print("RDK BPU 控制服务拒绝切换:", response)
    except (OSError, ValueError, json.JSONDecodeError) as exc:
        print("RDK BPU 控制服务不可用，尝试 SSH:", exc)
    remote = "sunrise@" + RDK_HOST
    command = (
        "pkill -f '^python3 .*rdk_bpu_policy_server.py' >/dev/null 2>&1 || true; "
        "sleep 1; "
        "nohup python3 " + shlex.quote(RDK_SERVER_SCRIPT) +
        " --model " + shlex.quote(RDK_HBM) +
        " --port " + str(BPU_PORT) +
        " >/tmp/microduck_bpu_server.log 2>&1 </dev/null &"
    )
    result = subprocess.run(
        ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", remote, "bash -lc " + shlex.quote(command)],
        capture_output=True, text=True,
    )
    if result.returncode:
        print("RDK 策略服务启动失败:", result.stderr.strip() or result.stdout.strip())
        return False
    print("RDK 任务策略服务已切换:", BPU_TASK, "->", RDK_HBM)
    return True

if not BPU_TASK:
    print("该任务目前没有可验证的任务专属 HBM；不会用其他任务视频替代。")
elif BPU_HBM is None or not BPU_HBM.is_file():
    print("Ubuntu 端 HBM 不存在:", BPU_HBM)
elif not BPU_VIDEO_SCRIPT.is_file():
    print("找不到板端闭环录制器:", BPU_VIDEO_SCRIPT)
elif BPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
    print("找不到 MotrixLab Python 环境:", MOTRIX_PYTHON)
elif BPU_TASK == "ball_balance" and (MOTRIX_RUNNER is None or not MOTRIX_RUNNER.is_file()):
    print("找不到 MotrixLab 启动器:", MOTRIX_RUNNER)
elif restart_rdk_policy_server():
    BPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    if BPU_TASK == "ball_balance":
        video_cmd = [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(BPU_VIDEO_SCRIPT)]
    else:
        video_cmd = [sys.executable, str(BPU_VIDEO_SCRIPT)]
    if BPU_TASK != "ball_balance":
        video_cmd += ["--task", BPU_TASK]
    video_cmd += [
        "--bpu-host", RDK_HOST, "--bpu-port", str(BPU_PORT),
        "--hbm", str(BPU_HBM), "--output", str(BPU_VIDEO_OUTPUT),
        "--steps", str(BPU_VIDEO_STEPS), "--width", str(BPU_VIDEO_WIDTH),
        "--height", str(BPU_VIDEO_HEIGHT),
    ]
    print("开始生成板端闭环视频:", " ".join(shlex.quote(x) for x in video_cmd))
    video_result = subprocess.run(
        video_cmd, cwd=PLAYGROUND_ROOT, env=os.environ.copy(),
        capture_output=True, text=True,
    )
    print("\n".join((video_result.stdout + "\n" + video_result.stderr).splitlines()[-35:]))
    if video_result.returncode != 0 or not BPU_VIDEO_OUTPUT.is_file():
        raise RuntimeError("BPU-in-the-loop 视频生成失败，请先查看上面最后 35 行日志。")
    BPU_REPORT = BPU_VIDEO_OUTPUT.with_suffix(".json")
    print("PASS-BPU-VIDEO:", BPU_VIDEO_OUTPUT)
    if BPU_REPORT.is_file():
        print(json.loads(BPU_REPORT.read_text(encoding="utf-8")))
    from IPython.display import Video, display
    display(Video(str(BPU_VIDEO_OUTPUT), embed=True))

SIM_COMMAND = 'python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11'
print("MuJoCo 回放命令模板:", SIM_COMMAND)

## 6. RDK X5 / BPU 审计

上一单元已经生成任务专属闭环视频。本单元只检查板端连通、HBM 元数据和服务日志；通用 MobileNet 样例只能证明系统运行时存在，不能代替任务视频。

In [ ]:
import time

RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
BPU_SMOKE_MODEL = os.getenv(
    "RDK_BPU_SMOKE_MODEL",
    "/opt/tros/humble/lib/dnn_benchmark_example/config/X5/mobilenetv1_224x224_nv12.bin",
)
BPU_SMOKE_INPUT_BYTES = int(os.getenv("RDK_BPU_SMOKE_INPUT_BYTES", "75264"))
BPU_HBM = os.getenv("RDK_BPU_HBM", "")
print("默认 BPU 样例模型:", BPU_SMOKE_MODEL)
print("默认 BPU 样例输入:", BPU_SMOKE_INPUT_BYTES, "bytes uint8")
print("MicroDuck 策略 HBM:", BPU_HBM or "该任务尚未提供")
probe_script = "for x in hrt_model_exec hb_mapper hrt_bin_dump hrt_bin_info python3; do if command -v $x >/dev/null 2>&1; then echo $x=$(command -v $x); else echo $x=MISSING; fi; done"
remote = "sunrise@" + RDK_HOST
probe = subprocess.run(["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", remote, "bash -lc " + shlex.quote(probe_script)], capture_output=True, text=True)
if probe.returncode:
    print("RDK 暂不可达，跳过板端验收:", probe.stderr.strip() or "ssh failed")
else:
    print("RDK 已连通:", RDK_HOST)
    print(probe.stdout)
    if not BPU_HBM:
        print("没有任务 HBM：不执行通用样例冒充任务推理。")
        smoke_model = BPU_SMOKE_MODEL
        smoke_bytes = BPU_SMOKE_INPUT_BYTES
        smoke_info_cmd = "hrt_model_exec model_info --model_file " + shlex.quote(smoke_model)
        smoke_info = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(smoke_info_cmd)], capture_output=True, text=True)
        print("BPU 样例 model_info exit:", smoke_info.returncode)
        print(smoke_info.stdout[-4000:])
        print(smoke_info.stderr[-1000:])
        if smoke_info.returncode == 0:
            import numpy as np
            local_input = OUTPUT_ROOT / "rdk_bpu_runtime_smoke.bin"
            np.zeros(smoke_bytes, dtype=np.uint8).tofile(local_input)
            remote_dir = "/tmp/microduck_notebook"
            remote_input = remote_dir + "/rdk_bpu_runtime_smoke.bin"
            subprocess.run(["ssh", remote, "mkdir", "-p", remote_dir], check=True)
            subprocess.run(["scp", str(local_input), remote + ":" + remote_input], check=True)
            dump_dir = remote_dir + "/runtime_dump"
            infer_cmd = "rm -rf " + shlex.quote(dump_dir) + " && mkdir -p " + shlex.quote(dump_dir) + " && hrt_model_exec infer --model_file " + shlex.quote(smoke_model) + " --input_file " + shlex.quote(remote_input) + " --enable_dump true --dump_format txt --dump_path " + shlex.quote(dump_dir)
            started = time.perf_counter()
            infer = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(infer_cmd)], capture_output=True, text=True)
            elapsed_ms = (time.perf_counter() - started) * 1000.0
            print("BPU runtime sample exit:", infer.returncode, "round-trip ms:", round(elapsed_ms, 2))
            print(infer.stdout[-4000:])
            print(infer.stderr[-1000:])
            if infer.returncode == 0:
                print("PASS-rdk-bpu-runtime-sample: X5 BPU 样例推理已返回 0。")
    else:
        info_cmd = "hrt_model_exec model_info --model_file " + shlex.quote(BPU_HBM)
        info = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(info_cmd)], capture_output=True, text=True)
        print("HBM model_info exit:", info.returncode)
        print(info.stdout[-6000:])
        print(info.stderr[-2000:])
        if info.returncode == 0:
            # hrt_model_exec consumes binary tensors; this zero observation is
            # only a transport/BPU smoke input, not a task-quality evaluation.
            import numpy as np
            local_input = OUTPUT_ROOT / "zero_obs_61_f32.bin"
            np.zeros((1, 61), dtype=np.float32).tofile(local_input)
            remote_dir = "/tmp/microduck_notebook"
            remote_input = remote_dir + "/zero_obs_61_f32.bin"
            subprocess.run(["ssh", remote, "mkdir", "-p", remote_dir], check=True)
            subprocess.run(["scp", str(local_input), remote + ":" + remote_input], check=True)
            dump_dir = remote_dir + "/dump"
            infer_cmd = "rm -rf " + shlex.quote(dump_dir) + " && mkdir -p " + shlex.quote(dump_dir) + " && hrt_model_exec infer --model_file " + shlex.quote(BPU_HBM) + " --input_file " + shlex.quote(remote_input) + " --enable_dump true --dump_format txt --dump_path " + shlex.quote(dump_dir)
            started = time.perf_counter()
            infer = subprocess.run(["ssh", remote, "bash -lc " + shlex.quote(infer_cmd)], capture_output=True, text=True)
            elapsed_ms = (time.perf_counter() - started) * 1000.0
            print("BPU infer exit:", infer.returncode, "round-trip ms:", round(elapsed_ms, 2))
            print(infer.stdout[-6000:])
            print(infer.stderr[-2000:])
            if infer.returncode == 0:
                print("PASS-rdk-bpu: hrt_model_exec infer 已返回 0。")
            else:
                print("未通过：优先以第 5 节 BPU-in-the-loop 报告为准。")

## 7. 直播结论

- `PASS-local-onnx`：ONNX checker 和本地 ONNX Runtime 通过。
- `PASS-BPU-VIDEO`：Ubuntu MuJoCo 视频中的每一个动作都由 RDK X5 对应 HBM 产生。
- `PASS-rdk-bpu-runtime-sample`：板端已安装的 X5 BPU 样例推理退出码为 0。
- `PASS-rdk-bpu`：SSH 连通、板端明确选择 BPU、MicroDuck HBM 推理命令退出码为 0，并记录输入输出形状与延迟。

没有 `PASS-BPU-VIDEO` 的任务，只能算“已有素材/接口说明”，不能算板端视频已完成。